MedMCQA: Delphi (GPT-OSS-120B organizer, Qwen3.6-35B-A3B, DeepSeek-V4 Flash)..


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/medmcqa/benchmark_subsets'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import os
import json
import hashlib
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Any, List, Tuple, Optional
import pandas as pd
import requests
from openai import OpenAI
from tqdm.auto import tqdm
from pathlib import Path
import getpass


In [ ]:
FOLDER_ID = ''
API_KEY = ''
LLM_API_URL = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'
CASE_COMPONENT_COLUMNS = ['question', 'option_a', 'option_b', 'option_c', 'option_d']
COLUMN_DISPLAY_NAMES: Dict[str, str] = {'question': 'Question', 'option_a': 'Option A', 'option_b': 'Option B', 'option_c': 'Option C', 'option_d': 'Option D'}
SAMPLE_SIZE = None
RANDOM_SEED = 42
REQUEST_TIMEOUT = 120
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
MAX_DELPHI_ROUNDS = 3
EARLY_STOP_ON_UNANIMITY = True
CASE_CONCURRENCY = 5
MIN_QUESTIONS_PER_EXPERT_R1 = 5
MAX_QUESTIONS_PER_EXPERT_R1 = 8
MAX_UNIFIED_QUESTIONS = 12
TEMPERATURE_ROUND1 = 0.3
TEMPERATURE_ROUND2 = 0.3
TEMPERATURE_ROUND3 = 0.3
TEMPERATURE_ORGANIZER = 0.3
TEMPERATURE_FINAL_SYNTHESIS = 0.3
MAX_TOKENS_ROUND1 = 10000
MAX_TOKENS_ROUND2 = 10000
MAX_TOKENS_ROUND3 = 10000
MAX_TOKENS_ORGANIZER = 10000
MAX_TOKENS_FINAL_SYNTHESIS = 10000
ANSWER_FIELD_NAME = 'selected_option'
ANSWER_LABELS = ['A', 'B', 'C', 'D']
PARTICIPANTS: List[Dict[str, str]] = [{'id': 'deepseekv4_default', 'provider': 'openai_compatible', 'model_uri': f'gpt://{FOLDER_ID}/deepseek-v4-flash/latest', 'role_name': 'Default', 'role_description': 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'}, {'id': 'gpt120b_default', 'provider': 'openai_compatible', 'model_uri': f'gpt://{FOLDER_ID}/gpt-oss-120b/latest', 'role_name': 'Default', 'role_description': 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'}, {'id': 'qwen36_default', 'provider': 'openai_compatible', 'model_uri': f'gpt://{FOLDER_ID}/qwen3.6-35b-a3b/latest', 'role_name': 'Default', 'role_description': 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'}]
ORGANIZER_PARTICIPANT_ID = 'gpt120b_default'
if ORGANIZER_PARTICIPANT_ID not in [p['id'] for p in PARTICIPANTS]:
    raise ValueError(f'ORGANIZER_PARTICIPANT_ID={ORGANIZER_PARTICIPANT_ID!r} not in PARTICIPANTS ids.')
participant_ids = [p['id'] for p in PARTICIPANTS]
if len(participant_ids) != len(set(participant_ids)):
    raise ValueError('Participant ids must be unique.')
PARTICIPANTS = [next((p for p in PARTICIPANTS if p['id'] == identifier)) for identifier in ['gpt120b_default', 'qwen36_default', 'deepseekv4_default']]
MODEL_MAX_OUTPUT_TOKENS = {'gpt-oss-120b': 10000, 'qwen3.6-35b-a3b': 10000, 'deepseek-v4-flash': 10000}


In [ ]:
def build_case_text(row_data: Dict[str, Any]) -> str:
    parts: List[str] = []
    for column in CASE_COMPONENT_COLUMNS:
        value = row_data.get(column)
        if pd.isna(value):
            continue
        value_str = str(value).strip()
        if not value_str or value_str.lower() == 'nan':
            continue
        label = COLUMN_DISPLAY_NAMES.get(column, column)
        parts.append(f'{label}:\n{value_str}')
    return '\n\n'.join(parts)


In [ ]:
from pathlib import Path
import getpass
import sqlite3
import threading
import math
from collections.abc import MutableMapping
from datetime import datetime, timezone
PACKAGE_NAME = 'shared_benchmark_runs'

def locate_notebook_directory():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        candidates = [parent, parent / 'benchmark_subsets', parent / 'experiments' / 'external_benchmarks/medmcqa' / 'benchmark_subsets', parent / 'experiments' / 'external_benchmarks/medmcqa']
        for candidate in candidates:
            if (candidate / PACKAGE_NAME / 'data_manifest.json').is_file():
                return candidate
    raise FileNotFoundError('Open this notebook within the project containing ' + PACKAGE_NAME)
NOTEBOOK_DIR = locate_notebook_directory()
PACKAGE_DIR = NOTEBOOK_DIR / PACKAGE_NAME
MANIFEST_PATH = PACKAGE_DIR / 'data_manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
SUBSET_ORDER = ['random_dev300', 'shao_hard100', 'goa100', 'marc100', 'marc250', 'dev4183']
assert MANIFEST['subset_order'] == SUBSET_ORDER
SUBSETS = {item['id']: item for item in MANIFEST['subsets']}
RUN_TAG = 'primary'
SKIP_COMPLETED = True
CHECKPOINT_BATCH_SIZE = 25
BUDGET_LIMIT_RUB = 98000.0
BUDGET_SAFETY_BUFFER_RUB = 1500.0
PRICES_RUB_PER_1000 = {'gpt-oss-120b': {'input': 0.3, 'output': 0.3}, 'qwen3.6-35b-a3b': {'input': 0.2, 'output': 0.3}, 'deepseek-v4-flash': {'input': 0.3, 'output': 0.5}}
PRICE_SOURCE = 'https://aistudio.yandex.ru/ru/docs/ai-studio/pricing'
RUNTIME_DIR = PACKAGE_DIR / 'runs' / RUN_TAG
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
LEDGER_PATH = RUNTIME_DIR / 'usage.sqlite3'

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def load_subset(subset_id):
    specification = SUBSETS[subset_id]
    path = PACKAGE_DIR / specification['csv']
    if sha256_file(path) != specification['sha256']:
        raise ValueError(f'Input checksum differs from the pinned manifest: {subset_id}')
    frame = pd.read_csv(path, sep=';', encoding='utf-8-sig', keep_default_na=False)
    if len(frame) != specification['n'] or frame['case_id'].duplicated().any():
        raise ValueError(f'Unexpected size or duplicate IDs: {subset_id}')
    required = ['case_id', *CASE_COMPONENT_COLUMNS, 'correct_option_letter']
    if any((column not in frame.columns for column in required)):
        raise ValueError(f'Missing required input columns: {subset_id}')
    if not frame['correct_option_letter'].isin(ANSWER_LABELS).all():
        raise ValueError(f'Invalid reference answer labels: {subset_id}')
    if frame[CASE_COMPONENT_COLUMNS].apply(lambda column: column.astype(str).str.strip().eq('')).any().any():
        raise ValueError(f'Empty question or answer option: {subset_id}')
    frame['case_text'] = frame.apply(lambda row: build_case_text(row.to_dict()), axis=1)
    return frame
print('Run sequence:', [(name, SUBSETS[name]['n']) for name in SUBSET_ORDER])
print('Total evaluations per system:', sum((SUBSETS[name]['n'] for name in SUBSET_ORDER)))
print('MARC origins:', {name: SUBSETS[name]['split_counts'] for name in ['marc100', 'marc250']})


In [ ]:
session = requests.Session()
llm_cache: Dict[str, Dict[str, Any]] = {}
openai_client = OpenAI(api_key=API_KEY, base_url='https://llm.api.cloud.yandex.net/v1', project=FOLDER_ID)

def make_cache_key(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> str:
    payload = {'provider': provider, 'model_uri': model_uri, 'system_text': system_text, 'user_text': user_text, 'temperature': temperature, 'max_tokens': max_tokens}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _build_token_dict(prompt_tokens: int=0, completion_tokens: int=0, total_tokens: int=0) -> Dict[str, int]:
    return {'prompt_tokens': int(prompt_tokens or 0), 'completion_tokens': int(completion_tokens or 0), 'total_tokens': int(total_tokens or 0)}

def _extract_openai_text(response: Any) -> str:
    if not getattr(response, 'choices', None):
        return ''
    message = response.choices[0].message
    content = getattr(message, 'content', '')
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
            elif hasattr(item, 'type') and getattr(item, 'type', None) == 'text':
                parts.append(getattr(item, 'text', ''))
        return '\n'.join([p for p in parts if p]).strip()
    return ''

def _extract_openai_usage(usage: Any) -> Dict[str, int]:
    if usage is None:
        return _build_token_dict()
    prompt_tokens = getattr(usage, 'prompt_tokens', None)
    completion_tokens = getattr(usage, 'completion_tokens', None)
    total_tokens = getattr(usage, 'total_tokens', None)
    if total_tokens is None:
        input_tokens = getattr(usage, 'input_tokens', None)
        output_tokens = getattr(usage, 'output_tokens', None)
        prompt_tokens = prompt_tokens if prompt_tokens is not None else input_tokens
        completion_tokens = completion_tokens if completion_tokens is not None else output_tokens
        total_tokens = int(prompt_tokens or 0) + int(completion_tokens or 0)
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def _extract_yandex_usage(result: Dict[str, Any]) -> Dict[str, int]:
    usage = result.get('usage', {}) or {}
    prompt_tokens = usage.get('inputTextTokens', 0) or 0
    completion_tokens = usage.get('completionTokens', 0) or 0
    total_tokens = usage.get('totalTokens', 0) or prompt_tokens + completion_tokens
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def call_yandex_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, timeout: int=REQUEST_TIMEOUT, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('yandex_rest', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        cached = llm_cache[cache_key]
        return {'text': cached['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    headers = {'Content-Type': 'application/json', 'Authorization': f'Api-Key {API_KEY}'}
    payload = {'modelUri': model_uri, 'completionOptions': {'stream': False, 'temperature': temperature, 'maxTokens': max_tokens}, 'messages': [{'role': 'system', 'text': system_text}, {'role': 'user', 'text': user_text}]}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = session.post(LLM_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.RequestException as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        if response.status_code == 200:
            data = response.json()
            result = data.get('result', {}) or {}
            raw_text = result.get('alternatives', [{}])[0].get('message', {}).get('text', '').strip()
            usage = _extract_yandex_usage(result)
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, 'prompt_tokens': usage['prompt_tokens'], 'completion_tokens': usage['completion_tokens'], 'total_tokens': usage['total_tokens'], 'cached': False}
        last_error_text = f'HTTP {response.status_code}: {response.text}'
        if response.status_code in (429, 500, 502, 503, 504):
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        raise RuntimeError(last_error_text)
    raise RuntimeError(f'Yandex REST call failed after {retries} retries. Last error: {last_error_text}')

def call_openai_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('openai_compatible', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        cached = llm_cache[cache_key]
        return {'text': cached['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = openai_client.chat.completions.create(model=model_uri, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens)
            raw_text = _extract_openai_text(response)
            usage = _extract_openai_usage(getattr(response, 'usage', None))
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, 'prompt_tokens': usage['prompt_tokens'], 'completion_tokens': usage['completion_tokens'], 'total_tokens': usage['total_tokens'], 'cached': False}
        except Exception as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
    raise RuntimeError(f'OpenAI-compatible call failed after {retries} retries. Last error: {last_error_text}')

def call_llm(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> Dict[str, Any]:
    if provider == 'yandex_rest':
        return call_yandex_llm(model_uri, system_text, user_text, temperature, max_tokens)
    if provider == 'openai_compatible':
        return call_openai_llm(model_uri, system_text, user_text, temperature, max_tokens)
    raise ValueError(f'Unknown provider: {provider}')


In [ ]:
def build_round1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are one of three expert medical exam reviewers participating anonymously in a Delphi consensus process.\nAt round 1 your task is to decompose a single-best-answer medical multiple-choice exam question (four options, exactly one correct) into a set of focused sub-questions, contribute\nextra considerations you believe are relevant beyond what is explicitly stated, and provide your\ninitial answer.\n\nReason carefully from the exam question. Do not invent facts absent from it.\nReturn JSON only.'.strip()

def build_round1_user_prompt(case_text: str) -> str:
    answer_list = json.dumps(ANSWER_LABELS)
    return f'Exam question:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nTasks:\n1) Decompose the exam question into {MIN_QUESTIONS_PER_EXPERT_R1}-{MAX_QUESTIONS_PER_EXPERT_R1} sub-questions that\n   together cover the underlying physiological or pathological mechanism, the reason each incorrect option is wrong, the single most discriminating fact that separates the correct option from the closest distractor, and any relevant guideline or textbook knowledge that resolves the question. Each sub-question must be answerable from the exam question content.\n2) For each sub-question give your initial answer and a brief rationale.\n3) List additional considerations you think are relevant but were not explicitly stated in the\n   exam question (e.g. easily-missed distinctions, ambiguous wording, edge cases).\n4) Give your initial answer: the option letter (A, B, C, or D) you currently believe is correct.\n   Confidence is your certainty in the option you just chose, rated 1-10. Rate 1-3 if the\n   exam question under-determines the answer, 4-7 if the chosen option is the best fit among\n   defensible alternatives, and 8-10 if the exam question explicitly determines it.\n\nReturn JSON only:\n{{\n  "sub_questions": [\n    {{"id": "Q1", "question": "...", "my_answer": "...", "rationale": "..."}}\n  ],\n  "additional_relevant_facts": ["..."],\n  "initial_answer": {{\n    "selected_option": "<one of {answer_list}>",\n    "confidence": 0\n  }}\n}}\n'.strip()

def build_organizer_compile_after_r1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are the Delphi process moderator. Three anonymous expert medical exam reviewers have just completed round 1\nand produced their decompositions and initial answers.\n\nYour duties:\n- Deduplicate and merge the sub-questions into a single unified questionnaire of at most\n  {MAX_UNIFIED_QUESTIONS} items. Prefer sub-questions raised by more experts; keep critical\n  ones even if raised by only one expert.\n- For each unified question, produce a short anonymous summary of how experts answered, and explicitly\n  list any extreme/outlier positions with their stated reasons.\n- Preserve all additional considerations contributed by experts.\n\nReturn JSON only.'.strip()

def build_organizer_compile_after_r1_user_prompt(case_text: str, expert_payloads: str) -> str:
    return f'Exam question:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nRound-1 raw expert payloads (anonymized as Expert A / B / C):\n<<<EXPERT_PAYLOADS>>>\n{expert_payloads}\n<<<END EXPERT_PAYLOADS>>>\n\nReturn JSON only in this exact format:\n{{\n  "unified_questions": [\n    {{\n      "id": "U1",\n      "question": "...",\n      "anonymous_summary_of_answers": "1-3 sentence summary of how experts answered",\n      "extreme_views": ["each outlier position with its stated reason"]\n    }}\n  ],\n  "merged_additional_considerations": ["..."],\n  "anonymous_overall_summary": "1 paragraph"\n}}\n\nHard constraints:\n- Number of unified_questions in [5, {MAX_UNIFIED_QUESTIONS}].\n- Output JSON only.'.strip()

def build_round_n_system_prompt(role_name: str, role_description: str, round_idx: int) -> str:
    return f"You are acting as {role_name}.\n{role_description}\n\nYou are one of three expert medical exam reviewers participating anonymously in a Delphi consensus process.\nThis is round {round_idx} of {MAX_DELPHI_ROUNDS}.\n\nYou will see a unified questionnaire compiled from all experts' previous-round answers, with an\nanonymous summary of others' positions and any extreme views with their reasons. Your task is to\nre-answer each question, indicate whether your answer has changed since the previous round and why,\nand update your answer.\n\nBe willing to revise. Be willing to keep your position if the exam question supports it. Return JSON only.".strip()

def build_round2_user_prompt(case_text: str, organizer_payload: str) -> str:
    answer_list = json.dumps(ANSWER_LABELS)
    return f'Exam question:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nUnified questionnaire and anonymous summary from round 1:\n<<<ORGANIZER_SUMMARY>>>\n{organizer_payload}\n<<<END ORGANIZER_SUMMARY>>>\n\nTasks:\n1) Answer each unified question. State whether your answer changed compared to your round-1 position\n   ("changed_from_previous": true/false) and explain why in one short sentence.\n2) Indicate whether the unified questionnaire is sufficient (covers everything important for this exam question).\n3) If not sufficient, list missing aspects that should be added in round 3.\n4) Update your answer: the option letter (A, B, C, or D) you currently believe is correct.\n   Confidence is your certainty in the option you just chose, rated 1-10. Rate 1-3 if the\n   exam question under-determines the answer, 4-7 if the chosen option is the best fit among\n   defensible alternatives, and 8-10 if the exam question explicitly determines it.\n\nReturn JSON only:\n{{\n  "answers": [\n    {{\n      "question_id": "U1",\n      "answer": "...",\n      "rationale": "...",\n      "changed_from_previous": true\n    }}\n  ],\n  "is_questionnaire_sufficient": true,\n  "missing_aspects": ["..."],\n  "current_answer": {{\n    "selected_option": "<one of {answer_list}>",\n    "confidence": 0\n  }}\n}}\n'.strip()

def build_organizer_compile_after_r2_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are the Delphi process moderator. Three anonymous expert medical exam reviewers have just completed round 2.\n\nYour duties for round 3:\n- Produce an anonymous summary of round-2 answers per unified question.\n- Explicitly highlight every outlier / extreme view (a position not shared by majority) with the reason\n  the dissenting expert provided. This is what experts must address in round 3.\n- Carry forward any missing aspects that experts flagged. You may add additional questions if\n  experts unanimously requested them.\n\nReturn JSON only.'.strip()

def build_organizer_compile_after_r2_user_prompt(case_text: str, organizer_summary_after_r1: str, expert_r2_payloads: str) -> str:
    return f'Exam question:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nOrganizer summary from round 1 (for continuity):\n<<<R1_SUMMARY>>>\n{organizer_summary_after_r1}\n<<<END R1_SUMMARY>>>\n\nRound-2 raw expert payloads (anonymized as Expert A / B / C):\n<<<R2_PAYLOADS>>>\n{expert_r2_payloads}\n<<<END R2_PAYLOADS>>>\n\nReturn JSON only:\n{{\n  "unified_questions_for_round3": [\n    {{\n      "id": "U1",\n      "question": "...",\n      "anonymous_summary_of_round2_answers": "...",\n      "extreme_views_with_reasons": ["explicit outlier-position + stated reason"]\n    }}\n  ],\n  "added_questions": ["any new questions, if all experts requested them"],\n  "anonymous_overall_summary": "1 paragraph"\n}}\n\nHard constraints:\n- Preserve all original unified question ids unless an expert majority requested removal.\n- Output JSON only.'.strip()

def build_round3_user_prompt(case_text: str, organizer_payload: str) -> str:
    answer_list = json.dumps(ANSWER_LABELS)
    return f'Exam question:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nUnified questionnaire from round 2, with anonymous summary and explicit extreme views with reasons:\n<<<ORGANIZER_SUMMARY>>>\n{organizer_payload}\n<<<END ORGANIZER_SUMMARY>>>\n\nFinal round duties:\n1) Provide your final answer to each question; mark whether you changed your view ("changed_from_previous").\n2) Explicitly review extreme views - list which ones you considered and whether they updated your stance.\n3) Rate the case on three axes, each 1-10:\n   - "answer_confidence": overall confidence that the selected option is correct.\n   - "distractor_elimination_quality": how convincingly the reasoning rules out the three incorrect options.\n   - "alignment_with_question_stem": how precisely the final answer matches exactly what the question stem asks, with no scope drift.\n4) Finalize your answer: the option letter (A, B, C, or D) you currently believe is correct.\n   Confidence is your certainty in the option you just chose, rated 1-10. Rate 1-3 if the\n   exam question under-determines the answer, 4-7 if the chosen option is the best fit among\n   defensible alternatives, and 8-10 if the exam question explicitly determines it.\n\nReturn JSON only:\n{{\n  "answers": [\n    {{\n      "question_id": "U1",\n      "answer": "...",\n      "rationale": "...",\n      "changed_from_previous": false\n    }}\n  ],\n  "outliers_addressed": [\n    {{"summary_of_outlier": "...", "updated_my_view": false, "reason": "..."}}\n  ],\n  "ratings": {{\n    "answer_confidence": 0,\n    "distractor_elimination_quality": 0,\n    "alignment_with_question_stem": 0\n  }},\n  "final_answer": {{\n    "selected_option": "<one of {answer_list}>",\n    "confidence": 0\n  }}\n}}\n'.strip()

def build_final_synthesis_system_prompt(role_name: str, role_description: str) -> str:
    return f"You are acting as {role_name}.\n{role_description}\n\nYou are the Delphi moderator. The expert panel has converged (or reached the round limit). You receive\nthe unified questionnaire, all experts' final answers, their ratings, and the consensus answer\n(already computed as the majority of expert votes on {ANSWER_FIELD_NAME}).\n\nCompose a single concise write-up that uses the consensus answer exactly, integrates the converged\nreasoning, and explicitly addresses any considerations that surfaced during the Delphi rounds.".strip()

def build_final_synthesis_user_prompt(case_text: str, consensus_answer: str, rounds_summary: str, final_answers_block: str, ratings_block: str, additional_considerations_block: str, author_id: str) -> str:
    return f'Exam question:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nConsensus answer (majority of final-round expert votes; use this verbatim):\n- Selected option: {consensus_answer}\n\nRounds summary (anonymous):\n<<<ROUNDS_SUMMARY>>>\n{rounds_summary}\n<<<END ROUNDS_SUMMARY>>>\n\nFinal-round expert answers per unified question (anonymous):\n<<<FINAL_ANSWERS>>>\n{final_answers_block}\n<<<END FINAL_ANSWERS>>>\n\nExpert ratings (answer_confidence / distractor_elimination_quality / alignment_with_question_stem):\n<<<RATINGS>>>\n{ratings_block}\n<<<END RATINGS>>>\n\nAdditional considerations surfaced during the rounds:\n<<<EXTRA>>>\n{additional_considerations_block}\n<<<END EXTRA>>>\n\nWrite the protocol in plain text. Same field labels and order:\n\nSelected option:\nRationale:\nProtocol Author: {author_id}\n\nHard requirements:\n- Selected option must equal exactly: {consensus_answer}\n- Rationale: 2-5 sentences synthesizing the converged reasoning from the rounds; note where consensus\n  differed and how it was resolved. Ground it only in the exam question content.\n'.strip()

def extract_first_json_object(text: str) -> Optional[Dict[str, Any]]:
    if text is None:
        return None
    raw = str(text).strip()
    try:
        return json.loads(raw)
    except Exception:
        pass
    match = re.search('\\{.*\\}', raw, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None
    return None

def _coerce_str(value: Any) -> str:
    if value is None:
        return ''
    return str(value).strip()

def _normalize_label(value: Any, allowed: List[str]) -> str:
    if value is None:
        return ''
    s = str(value).strip()
    if not s:
        return ''
    lookup = {a.lower(): a for a in allowed}
    return lookup.get(s.lower(), '')

def _int_in_range(value: Any, lo: int, hi: int, default: int) -> int:
    try:
        v = int(value)
    except Exception:
        return default
    return max(lo, min(hi, v))

def parse_round1_output(text: str) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    sub_qs_raw = data.get('sub_questions') or []
    sub_qs: List[Dict[str, str]] = []
    if isinstance(sub_qs_raw, list):
        for idx, item in enumerate(sub_qs_raw, start=1):
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('id')) or f'Q{idx}'
            question = _coerce_str(item.get('question'))
            if not question:
                continue
            sub_qs.append({'id': qid, 'question': question, 'my_answer': _coerce_str(item.get('my_answer')), 'rationale': _coerce_str(item.get('rationale'))})
    facts_raw = data.get('additional_relevant_facts') or []
    if not isinstance(facts_raw, list):
        facts_raw = [_coerce_str(facts_raw)]
    facts = [_coerce_str(x) for x in facts_raw if _coerce_str(x)]
    ans = data.get('initial_answer') or {}
    return {'sub_questions': sub_qs, 'additional_relevant_facts': facts, 'answer': {ANSWER_FIELD_NAME: _normalize_label(ans.get(ANSWER_FIELD_NAME), ANSWER_LABELS), 'confidence': _int_in_range(ans.get('confidence'), 1, 10, 5)}, 'raw_text': text}

def parse_compile_after_r1_output(text: str) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    unified_raw = data.get('unified_questions') or []
    unified: List[Dict[str, Any]] = []
    if isinstance(unified_raw, list):
        for idx, item in enumerate(unified_raw, start=1):
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('id')) or f'U{idx}'
            question = _coerce_str(item.get('question'))
            if not question:
                continue
            extremes = item.get('extreme_views') or []
            if not isinstance(extremes, list):
                extremes = [_coerce_str(extremes)]
            extremes = [_coerce_str(x) for x in extremes if _coerce_str(x)]
            unified.append({'id': qid, 'question': question, 'anonymous_summary_of_answers': _coerce_str(item.get('anonymous_summary_of_answers')), 'extreme_views': extremes})
    extras_raw = data.get('merged_additional_considerations') or []
    if not isinstance(extras_raw, list):
        extras_raw = [_coerce_str(extras_raw)]
    extras = [_coerce_str(x) for x in extras_raw if _coerce_str(x)]
    return {'unified_questions': unified, 'merged_additional_considerations': extras, 'anonymous_overall_summary': _coerce_str(data.get('anonymous_overall_summary')), 'raw_text': text}

def parse_round_n_output(text: str, round_idx: int) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    answers_raw = data.get('answers') or []
    answers: List[Dict[str, Any]] = []
    if isinstance(answers_raw, list):
        for item in answers_raw:
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('question_id'))
            if not qid:
                continue
            answers.append({'question_id': qid, 'answer': _coerce_str(item.get('answer')), 'rationale': _coerce_str(item.get('rationale')), 'changed_from_previous': bool(item.get('changed_from_previous'))})
    missing_raw = data.get('missing_aspects') or []
    if not isinstance(missing_raw, list):
        missing_raw = [_coerce_str(missing_raw)]
    missing = [_coerce_str(x) for x in missing_raw if _coerce_str(x)]
    ans_key = 'current_answer' if round_idx == 2 else 'final_answer'
    ans = data.get(ans_key) or data.get('current_answer') or data.get('final_answer') or {}
    return {'round_idx': round_idx, 'answers': answers, 'is_questionnaire_sufficient': bool(data.get('is_questionnaire_sufficient', True)), 'missing_aspects': missing, 'answer': {ANSWER_FIELD_NAME: _normalize_label(ans.get(ANSWER_FIELD_NAME), ANSWER_LABELS), 'confidence': _int_in_range(ans.get('confidence'), 1, 10, 5)}, 'raw_text': text}

def parse_compile_after_r2_output(text: str) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    unified_raw = data.get('unified_questions_for_round3') or []
    unified: List[Dict[str, Any]] = []
    if isinstance(unified_raw, list):
        for idx, item in enumerate(unified_raw, start=1):
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('id')) or f'U{idx}'
            question = _coerce_str(item.get('question'))
            if not question:
                continue
            extremes = item.get('extreme_views_with_reasons') or []
            if not isinstance(extremes, list):
                extremes = [_coerce_str(extremes)]
            extremes = [_coerce_str(x) for x in extremes if _coerce_str(x)]
            unified.append({'id': qid, 'question': question, 'anonymous_summary_of_round2_answers': _coerce_str(item.get('anonymous_summary_of_round2_answers')), 'extreme_views_with_reasons': extremes})
    added_raw = data.get('added_questions') or []
    if not isinstance(added_raw, list):
        added_raw = [_coerce_str(added_raw)]
    added = [_coerce_str(x) for x in added_raw if _coerce_str(x)]
    return {'unified_questions_for_round3': unified, 'added_questions': added, 'anonymous_overall_summary': _coerce_str(data.get('anonymous_overall_summary')), 'raw_text': text}

def parse_round3_output(text: str) -> Dict[str, Any]:
    base = parse_round_n_output(text, 3)
    data = extract_first_json_object(text) or {}
    outliers_raw = data.get('outliers_addressed') or []
    outliers: List[Dict[str, Any]] = []
    if isinstance(outliers_raw, list):
        for item in outliers_raw:
            if not isinstance(item, dict):
                continue
            outliers.append({'summary_of_outlier': _coerce_str(item.get('summary_of_outlier')), 'updated_my_view': bool(item.get('updated_my_view')), 'reason': _coerce_str(item.get('reason'))})
    ratings_raw = data.get('ratings') or {}
    ratings = {'answer_confidence': _int_in_range(ratings_raw.get('answer_confidence'), 1, 10, 5), 'distractor_elimination_quality': _int_in_range(ratings_raw.get('distractor_elimination_quality'), 1, 10, 5), 'alignment_with_question_stem': _int_in_range(ratings_raw.get('alignment_with_question_stem'), 1, 10, 5)}
    base['outliers_addressed'] = outliers
    base['ratings'] = ratings
    return base

def extract_field_value(text: str, field: str) -> str:
    if text is None:
        return ''
    m = re.search(f'{re.escape(field)}\\s*:\\s*(.+)', str(text), re.IGNORECASE)
    return m.group(1).strip() if m else ''

def extract_final_answer(protocol_text: str) -> str:
    return _normalize_label(extract_field_value(protocol_text, 'Selected option'), ANSWER_LABELS)


In [ ]:
def _format_anonymous_label(idx: int) -> str:
    return f"Expert {chr(ord('A') + idx)}"

def _format_round1_payloads(round1_outputs: List[Dict[str, Any]]) -> str:
    chunks: List[str] = []
    for idx, out in enumerate(round1_outputs):
        chunks.append(f'### {_format_anonymous_label(idx)}')
        chunks.append('Initial answer:')
        ans = out['parsed']['answer']
        chunks.append(f'  {ANSWER_FIELD_NAME}: {ans[ANSWER_FIELD_NAME]}')
        chunks.append(f"  confidence: {ans['confidence']}")
        chunks.append('Sub-questions and initial answers:')
        for q in out['parsed']['sub_questions']:
            chunks.append(f"  [{q['id']}] Q: {q['question']}")
            chunks.append(f"      A: {q['my_answer']}")
            if q['rationale']:
                chunks.append(f"      rationale: {q['rationale']}")
        if out['parsed']['additional_relevant_facts']:
            chunks.append('Additional considerations:')
            for f in out['parsed']['additional_relevant_facts']:
                chunks.append(f'  - {f}')
        chunks.append('')
    return '\n'.join(chunks).strip()

def _format_round_n_payloads(round_outputs: List[Dict[str, Any]], round_idx: int) -> str:
    chunks: List[str] = []
    for idx, out in enumerate(round_outputs):
        chunks.append(f'### {_format_anonymous_label(idx)}')
        ans = out['parsed'].get('answer') or {ANSWER_FIELD_NAME: '', 'confidence': 0}
        chunks.append('Answer at this round:')
        chunks.append(f"  {ANSWER_FIELD_NAME}: {ans.get(ANSWER_FIELD_NAME, '')}")
        chunks.append(f"  confidence: {ans.get('confidence', 0)}")
        chunks.append(f"is_questionnaire_sufficient: {out['parsed'].get('is_questionnaire_sufficient', True)}")
        if out['parsed'].get('missing_aspects'):
            chunks.append('missing_aspects:')
            for m in out['parsed']['missing_aspects']:
                chunks.append(f'  - {m}')
        chunks.append('Answers:')
        for a in out['parsed'].get('answers', []):
            chunks.append(f"  [{a['question_id']}] A: {a['answer']}")
            if a['rationale']:
                chunks.append(f"      rationale: {a['rationale']}")
            chunks.append(f"      changed_from_previous: {a['changed_from_previous']}")
        if round_idx == 3 and 'ratings' in out['parsed']:
            r = out['parsed']['ratings']
            chunks.append('Ratings:')
            chunks.append(f"  answer_confidence: {r['answer_confidence']}")
            chunks.append(f"  distractor_elimination_quality: {r['distractor_elimination_quality']}")
            chunks.append(f"  alignment_with_question_stem: {r['alignment_with_question_stem']}")
            if out['parsed'].get('outliers_addressed'):
                chunks.append('outliers_addressed:')
                for o in out['parsed']['outliers_addressed']:
                    chunks.append(f"  - outlier: {o['summary_of_outlier']}; updated_my_view: {o['updated_my_view']}; reason: {o['reason']}")
        chunks.append('')
    return '\n'.join(chunks).strip()

def _format_organizer_payload_for_expert(compiled: Dict[str, Any], variant: str) -> str:
    chunks: List[str] = []
    if variant == 'after_r1':
        chunks.append('Unified questionnaire (round 2 input):')
        for q in compiled.get('unified_questions', []):
            chunks.append(f"  [{q['id']}] {q['question']}")
            if q['anonymous_summary_of_answers']:
                chunks.append(f"      anonymous summary: {q['anonymous_summary_of_answers']}")
            if q['extreme_views']:
                chunks.append('      extreme views:')
                for ev in q['extreme_views']:
                    chunks.append(f'        * {ev}')
        if compiled.get('merged_additional_considerations'):
            chunks.append('Additional considerations to keep in mind:')
            for c in compiled['merged_additional_considerations']:
                chunks.append(f'  - {c}')
        if compiled.get('anonymous_overall_summary'):
            chunks.append(f"Overall summary: {compiled['anonymous_overall_summary']}")
    else:
        chunks.append('Unified questionnaire (round 3 input) with extreme views:')
        for q in compiled.get('unified_questions_for_round3', []):
            chunks.append(f"  [{q['id']}] {q['question']}")
            if q['anonymous_summary_of_round2_answers']:
                chunks.append(f"      anonymous summary: {q['anonymous_summary_of_round2_answers']}")
            if q['extreme_views_with_reasons']:
                chunks.append('      extreme views with reasons:')
                for ev in q['extreme_views_with_reasons']:
                    chunks.append(f'        * {ev}')
        if compiled.get('added_questions'):
            chunks.append('Added questions (unanimously requested):')
            for q in compiled['added_questions']:
                chunks.append(f'  - {q}')
        if compiled.get('anonymous_overall_summary'):
            chunks.append(f"Overall summary: {compiled['anonymous_overall_summary']}")
    return '\n'.join(chunks)

def run_round1(case_text: str, case_participants: List[Dict[str, str]]) -> List[Dict[str, Any]]:

    def _call_one(participant: Dict[str, str]) -> Dict[str, Any]:
        system_text = build_round1_system_prompt(participant['role_name'], participant['role_description'])
        user_text = build_round1_user_prompt(case_text)
        response = call_llm(provider=participant['provider'], model_uri=participant['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_ROUND1, max_tokens=MAX_TOKENS_ROUND1)
        print('DELPHI R1 EXPERT:', participant['id'])
        parsed = parse_round1_output(response['text'])
        return {'participant_id': participant['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'parsed': parsed, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}
    with ThreadPoolExecutor(max_workers=len(case_participants)) as pool:
        outputs = list(pool.map(_call_one, case_participants))
    return outputs

def run_organizer_compile_after_r1(case_text: str, round1_outputs: List[Dict[str, Any]], case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    organizer = next((p for p in case_participants if p['id'] == ORGANIZER_PARTICIPANT_ID), None)
    if organizer is None:
        raise RuntimeError(f'Organizer {ORGANIZER_PARTICIPANT_ID!r} missing in case participants.')
    expert_payloads = _format_round1_payloads(round1_outputs)
    system_text = build_organizer_compile_after_r1_system_prompt(organizer['role_name'], organizer['role_description'])
    user_text = build_organizer_compile_after_r1_user_prompt(case_text, expert_payloads)
    response = call_llm(provider=organizer['provider'], model_uri=organizer['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_ORGANIZER, max_tokens=MAX_TOKENS_ORGANIZER)
    print('DELPHI ORGANIZER COMPILE AFTER R1')
    compiled = parse_compile_after_r1_output(response['text'])
    return {'organizer_id': organizer['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'compiled': compiled, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}

def run_round_n(case_text: str, organizer_payload: str, round_idx: int, case_participants: List[Dict[str, str]]) -> List[Dict[str, Any]]:

    def _call_one(participant: Dict[str, str]) -> Dict[str, Any]:
        system_text = build_round_n_system_prompt(participant['role_name'], participant['role_description'], round_idx)
        if round_idx == 2:
            user_text = build_round2_user_prompt(case_text, organizer_payload)
            max_tokens = MAX_TOKENS_ROUND2
            temp = TEMPERATURE_ROUND2
        elif round_idx == 3:
            user_text = build_round3_user_prompt(case_text, organizer_payload)
            max_tokens = MAX_TOKENS_ROUND3
            temp = TEMPERATURE_ROUND3
        else:
            raise ValueError(f'Unsupported round_idx {round_idx} for run_round_n')
        response = call_llm(provider=participant['provider'], model_uri=participant['model_uri'], system_text=system_text, user_text=user_text, temperature=temp, max_tokens=max_tokens)
        print(f'DELPHI R{round_idx} EXPERT:', participant['id'])
        if round_idx == 3:
            parsed = parse_round3_output(response['text'])
        else:
            parsed = parse_round_n_output(response['text'], round_idx)
        return {'participant_id': participant['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'parsed': parsed, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}
    with ThreadPoolExecutor(max_workers=len(case_participants)) as pool:
        outputs = list(pool.map(_call_one, case_participants))
    return outputs

def run_organizer_compile_after_r2(case_text: str, organizer_after_r1_compiled: Dict[str, Any], round2_outputs: List[Dict[str, Any]], case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    organizer = next((p for p in case_participants if p['id'] == ORGANIZER_PARTICIPANT_ID), None)
    if organizer is None:
        raise RuntimeError(f'Organizer {ORGANIZER_PARTICIPANT_ID!r} missing in case participants.')
    r1_summary_text = _format_organizer_payload_for_expert(organizer_after_r1_compiled, 'after_r1')
    r2_payloads = _format_round_n_payloads(round2_outputs, 2)
    system_text = build_organizer_compile_after_r2_system_prompt(organizer['role_name'], organizer['role_description'])
    user_text = build_organizer_compile_after_r2_user_prompt(case_text, r1_summary_text, r2_payloads)
    response = call_llm(provider=organizer['provider'], model_uri=organizer['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_ORGANIZER, max_tokens=MAX_TOKENS_ORGANIZER)
    print('DELPHI ORGANIZER COMPILE AFTER R2')
    compiled = parse_compile_after_r2_output(response['text'])
    return {'organizer_id': organizer['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'compiled': compiled, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}

def _answers_unanimous(per_expert: List[Dict[str, Any]]) -> bool:
    values = {e['answer'][ANSWER_FIELD_NAME] for e in per_expert if e['answer'][ANSWER_FIELD_NAME]}
    return len(values) == 1

def compute_delphi_consensus(final_round_outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    answers = [o['parsed']['answer'] for o in final_round_outputs]
    counts = Counter((a[ANSWER_FIELD_NAME] for a in answers if a[ANSWER_FIELD_NAME]))
    moderator_answer = next((o['parsed']['answer'] for o in final_round_outputs if o['participant_id'] == ORGANIZER_PARTICIPANT_ID), None)

    def _pick(counts: Counter, field: str, fallback: str) -> str:
        if not counts:
            return fallback
        top_count = counts.most_common(1)[0][1]
        tied = [label for label, c in counts.items() if c == top_count]
        if len(tied) == 1:
            return tied[0]
        moderator_label = moderator_answer.get(field) if moderator_answer else None
        if moderator_label in tied:
            return moderator_label
        confidence_by_label = {label: 0.0 for label in tied}
        for a in answers:
            label = a.get(field)
            if label in confidence_by_label:
                conf = a.get('confidence')
                if isinstance(conf, (int, float)):
                    confidence_by_label[label] += float(conf)
        return max(tied, key=lambda label: confidence_by_label[label])
    consensus_answer = _pick(counts, ANSWER_FIELD_NAME, ANSWER_LABELS[0])
    return {'consensus_answer': consensus_answer, 'answer_counts': dict(counts)}

def _aggregate_ratings(round3_outputs: List[Dict[str, Any]]) -> Dict[str, float]:
    if not round3_outputs:
        return {}
    keys = ['answer_confidence', 'distractor_elimination_quality', 'alignment_with_question_stem']
    out: Dict[str, float] = {}
    for k in keys:
        values = []
        for o in round3_outputs:
            r = o['parsed'].get('ratings') or {}
            v = r.get(k)
            if isinstance(v, (int, float)):
                values.append(float(v))
        out[k + '_mean'] = round(sum(values) / len(values), 2) if values else 0.0
        out[k + '_median'] = round(sorted(values)[len(values) // 2], 2) if values else 0.0
    return out

def _r1_outputs_as_answer_shape(round1_outputs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    converted: List[Dict[str, Any]] = []
    for o in round1_outputs:
        parsed = o['parsed']
        answers = [{'question_id': q['id'], 'answer': q['my_answer'], 'rationale': q['rationale'], 'changed_from_previous': False} for q in parsed.get('sub_questions', [])]
        converted.append({'participant_id': o['participant_id'], 'parsed': {'answers': answers, 'answer': parsed['answer'], 'is_questionnaire_sufficient': True, 'missing_aspects': []}})
    return converted

def _format_final_answers_block(final_outputs: List[Dict[str, Any]]) -> str:
    return _format_round_n_payloads(final_outputs, 3)

def _format_ratings_block(final_outputs: List[Dict[str, Any]], aggregate: Dict[str, float]) -> str:
    lines: List[str] = []
    for idx, o in enumerate(final_outputs):
        r = o['parsed'].get('ratings') or {}
        lines.append(f"  {_format_anonymous_label(idx)}: answer_confidence={r.get('answer_confidence')}, distractor_elimination_quality={r.get('distractor_elimination_quality')}, alignment_with_question_stem={r.get('alignment_with_question_stem')}")
    if aggregate:
        lines.append('  Aggregate (mean / median):')
        for k in ['answer_confidence', 'distractor_elimination_quality', 'alignment_with_question_stem']:
            lines.append(f"    {k}: mean={aggregate.get(k + '_mean')}, median={aggregate.get(k + '_median')}")
    return '\n'.join(lines)

def _format_rounds_summary(organizer_after_r1: Dict[str, Any], organizer_after_r2: Optional[Dict[str, Any]]) -> str:
    chunks: List[str] = []
    chunks.append('Round 1 organizer summary:')
    chunks.append(organizer_after_r1['compiled'].get('anonymous_overall_summary', '(empty)'))
    if organizer_after_r2 is not None:
        chunks.append('')
        chunks.append('Round 2 organizer summary:')
        chunks.append(organizer_after_r2['compiled'].get('anonymous_overall_summary', '(empty)'))
    return '\n'.join(chunks)

def _format_additional_considerations(organizer_after_r1: Dict[str, Any], organizer_after_r2: Optional[Dict[str, Any]]) -> str:
    items: List[str] = []
    for source in (organizer_after_r1, organizer_after_r2):
        if source is None:
            continue
        compiled = source['compiled']
        for c in compiled.get('merged_additional_considerations', []) or []:
            if c and c not in items:
                items.append(c)
        for q in compiled.get('unified_questions') or compiled.get('unified_questions_for_round3') or []:
            for ev in q.get('extreme_views') or q.get('extreme_views_with_reasons') or []:
                if ev and ev not in items:
                    items.append(ev)
    if not items:
        return '(none)'
    return '\n'.join((f'- {x}' for x in items))

def run_final_synthesis(case_text: str, consensus: Dict[str, Any], final_outputs: List[Dict[str, Any]], rounds_aggregate_ratings: Dict[str, float], organizer_after_r1: Dict[str, Any], organizer_after_r2: Optional[Dict[str, Any]], case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    organizer = next((p for p in case_participants if p['id'] == ORGANIZER_PARTICIPANT_ID), None)
    if organizer is None:
        raise RuntimeError(f'Organizer {ORGANIZER_PARTICIPANT_ID!r} missing in case participants.')
    rounds_summary = _format_rounds_summary(organizer_after_r1, organizer_after_r2)
    final_answers_block = _format_final_answers_block(final_outputs)
    ratings_block = _format_ratings_block(final_outputs, rounds_aggregate_ratings)
    additional_considerations_block = _format_additional_considerations(organizer_after_r1, organizer_after_r2)
    system_text = build_final_synthesis_system_prompt(organizer['role_name'], organizer['role_description'])
    user_text = build_final_synthesis_user_prompt(case_text=case_text, consensus_answer=consensus['consensus_answer'], rounds_summary=rounds_summary, final_answers_block=final_answers_block, ratings_block=ratings_block, additional_considerations_block=additional_considerations_block, author_id='delphi_consensus')
    response = call_llm(provider=organizer['provider'], model_uri=organizer['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_FINAL_SYNTHESIS, max_tokens=MAX_TOKENS_FINAL_SYNTHESIS)
    print('DELPHI FINAL SYNTHESIS')
    return {'organizer_id': organizer['id'], 'organizer_model_uri': organizer['model_uri'], 'organizer_provider': organizer['provider'], 'organizer_role_name': organizer['role_name'], 'organizer_role_description': organizer['role_description'], 'system_prompt': system_text, 'user_prompt': user_text, 'final_protocol': response['text'].strip(), 'final_answer': extract_final_answer(response['text']), 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}

def run_delphi_for_case(case_text: str, case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    round1_outputs = run_round1(case_text, case_participants)
    organizer_after_r1 = run_organizer_compile_after_r1(case_text, round1_outputs, case_participants)
    rounds_executed = 1
    final_round_outputs = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round1_outputs]
    organizer_after_r2: Optional[Dict[str, Any]] = None
    early_stop_reason: Optional[str] = None
    round2_outputs: List[Dict[str, Any]] = []
    round3_outputs: List[Dict[str, Any]] = []
    if EARLY_STOP_ON_UNANIMITY and _answers_unanimous([o['parsed'] for o in round1_outputs]):
        early_stop_reason = 'unanimous_after_r1'
    else:
        organizer_payload = _format_organizer_payload_for_expert(organizer_after_r1['compiled'], 'after_r1')
        round2_outputs = run_round_n(case_text, organizer_payload, 2, case_participants)
        rounds_executed = 2
        final_round_outputs = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round2_outputs]
        if EARLY_STOP_ON_UNANIMITY and _answers_unanimous([o['parsed'] for o in round2_outputs]):
            early_stop_reason = 'unanimous_after_r2'
        elif MAX_DELPHI_ROUNDS >= 3:
            organizer_after_r2 = run_organizer_compile_after_r2(case_text, organizer_after_r1['compiled'], round2_outputs, case_participants)
            organizer_payload_r3 = _format_organizer_payload_for_expert(organizer_after_r2['compiled'], 'after_r2')
            round3_outputs = run_round_n(case_text, organizer_payload_r3, 3, case_participants)
            rounds_executed = 3
            final_round_outputs = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round3_outputs]
    consensus = compute_delphi_consensus(final_round_outputs)
    rounds_aggregate_ratings = _aggregate_ratings(round3_outputs) if round3_outputs else {}
    if round3_outputs:
        final_outputs_for_synth = round3_outputs
    elif round2_outputs:
        final_outputs_for_synth = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round2_outputs]
    else:
        final_outputs_for_synth = _r1_outputs_as_answer_shape(round1_outputs)
    final_synth = run_final_synthesis(case_text=case_text, consensus=consensus, final_outputs=final_outputs_for_synth, rounds_aggregate_ratings=rounds_aggregate_ratings, organizer_after_r1=organizer_after_r1, organizer_after_r2=organizer_after_r2, case_participants=case_participants)
    return {'round1_outputs': round1_outputs, 'organizer_after_r1': organizer_after_r1, 'round2_outputs': round2_outputs, 'organizer_after_r2': organizer_after_r2, 'round3_outputs': round3_outputs, 'consensus': consensus, 'ratings_aggregate': rounds_aggregate_ratings, 'rounds_executed': rounds_executed, 'early_stop_reason': early_stop_reason, 'final_synthesis': final_synth}


In [ ]:
def run_delphi_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:

    def _process_one_case(row_data: Dict[str, Any]) -> Dict[str, Any]:
        row_data = dict(row_data)
        case_text = row_data['case_text']
        case_participants = [p.copy() for p in PARTICIPANTS]
        delphi = run_delphi_for_case(case_text, case_participants)
        for participant in case_participants:
            pid = participant['id']
            row_data[f'participant_provider__{pid}'] = participant['provider']
            row_data[f'participant_model_uri__{pid}'] = participant['model_uri']
            row_data[f'participant_role_name__{pid}'] = participant['role_name']
            row_data[f'participant_role_description__{pid}'] = participant['role_description']
            r1 = next((o for o in delphi['round1_outputs'] if o['participant_id'] == pid), None)
            if r1 is not None:
                ans = r1['parsed']['answer']
                row_data[f'r1_{ANSWER_FIELD_NAME}__{pid}'] = ans[ANSWER_FIELD_NAME]
                row_data[f'r1_confidence__{pid}'] = ans['confidence']
                row_data[f'r1_n_subquestions__{pid}'] = len(r1['parsed']['sub_questions'])
                row_data[f'r1_raw__{pid}'] = r1['raw_text']
                row_data[f'r1_prompt_tokens__{pid}'] = r1['prompt_tokens']
                row_data[f'r1_completion_tokens__{pid}'] = r1['completion_tokens']
                row_data[f'r1_total_tokens__{pid}'] = r1['total_tokens']
                row_data[f'r1_cached__{pid}'] = r1['cached']
            r2 = next((o for o in delphi['round2_outputs'] if o['participant_id'] == pid), None)
            if r2 is not None:
                ans = r2['parsed']['answer']
                row_data[f'r2_{ANSWER_FIELD_NAME}__{pid}'] = ans[ANSWER_FIELD_NAME]
                row_data[f'r2_confidence__{pid}'] = ans['confidence']
                row_data[f'r2_n_answers__{pid}'] = len(r2['parsed']['answers'])
                row_data[f'r2_changed_count__{pid}'] = sum((1 for a in r2['parsed']['answers'] if a['changed_from_previous']))
                row_data[f'r2_is_sufficient__{pid}'] = r2['parsed']['is_questionnaire_sufficient']
                row_data[f'r2_raw__{pid}'] = r2['raw_text']
                row_data[f'r2_prompt_tokens__{pid}'] = r2['prompt_tokens']
                row_data[f'r2_completion_tokens__{pid}'] = r2['completion_tokens']
                row_data[f'r2_total_tokens__{pid}'] = r2['total_tokens']
                row_data[f'r2_cached__{pid}'] = r2['cached']
            r3 = next((o for o in delphi['round3_outputs'] if o['participant_id'] == pid), None)
            if r3 is not None:
                ans = r3['parsed']['answer']
                row_data[f'r3_{ANSWER_FIELD_NAME}__{pid}'] = ans[ANSWER_FIELD_NAME]
                row_data[f'r3_confidence__{pid}'] = ans['confidence']
                row_data[f'r3_n_answers__{pid}'] = len(r3['parsed']['answers'])
                row_data[f'r3_changed_count__{pid}'] = sum((1 for a in r3['parsed']['answers'] if a['changed_from_previous']))
                ratings = r3['parsed'].get('ratings') or {}
                row_data[f'r3_rating_answer_confidence__{pid}'] = ratings.get('answer_confidence')
                row_data[f'r3_rating_distractor_elimination_quality__{pid}'] = ratings.get('distractor_elimination_quality')
                row_data[f'r3_rating_alignment_with_question_stem__{pid}'] = ratings.get('alignment_with_question_stem')
                row_data[f'r3_outliers_addressed_count__{pid}'] = len(r3['parsed'].get('outliers_addressed', []))
                row_data[f'r3_raw__{pid}'] = r3['raw_text']
                row_data[f'r3_prompt_tokens__{pid}'] = r3['prompt_tokens']
                row_data[f'r3_completion_tokens__{pid}'] = r3['completion_tokens']
                row_data[f'r3_total_tokens__{pid}'] = r3['total_tokens']
                row_data[f'r3_cached__{pid}'] = r3['cached']
        org_r1 = delphi['organizer_after_r1']
        row_data['organizer_after_r1_raw'] = org_r1['raw_text']
        row_data['organizer_after_r1_prompt_tokens'] = org_r1['prompt_tokens']
        row_data['organizer_after_r1_completion_tokens'] = org_r1['completion_tokens']
        row_data['organizer_after_r1_total_tokens'] = org_r1['total_tokens']
        row_data['organizer_after_r1_cached'] = org_r1['cached']
        row_data['organizer_after_r1_n_unified_questions'] = len(org_r1['compiled'].get('unified_questions', []))
        org_r2 = delphi['organizer_after_r2']
        if org_r2 is not None:
            row_data['organizer_after_r2_raw'] = org_r2['raw_text']
            row_data['organizer_after_r2_prompt_tokens'] = org_r2['prompt_tokens']
            row_data['organizer_after_r2_completion_tokens'] = org_r2['completion_tokens']
            row_data['organizer_after_r2_total_tokens'] = org_r2['total_tokens']
            row_data['organizer_after_r2_cached'] = org_r2['cached']
            row_data['organizer_after_r2_n_unified_questions'] = len(org_r2['compiled'].get('unified_questions_for_round3', []))
        else:
            row_data['organizer_after_r2_prompt_tokens'] = 0
            row_data['organizer_after_r2_completion_tokens'] = 0
            row_data['organizer_after_r2_total_tokens'] = 0
        row_data['delphi_rounds_executed'] = delphi['rounds_executed']
        row_data['delphi_early_stop_reason'] = delphi['early_stop_reason'] or ''
        row_data[f'delphi_consensus_{ANSWER_FIELD_NAME}'] = delphi['consensus']['consensus_answer']
        row_data['delphi_answer_counts_json'] = json.dumps(delphi['consensus']['answer_counts'], ensure_ascii=False)
        for k, v in (delphi['ratings_aggregate'] or {}).items():
            row_data[f'delphi_ratings_{k}'] = v
        synth = delphi['final_synthesis']
        row_data['final_synthesis_prompt_tokens'] = synth['prompt_tokens']
        row_data['final_synthesis_completion_tokens'] = synth['completion_tokens']
        row_data['final_synthesis_total_tokens'] = synth['total_tokens']
        row_data['final_synthesis_cached'] = synth['cached']
        row_data['winner_id'] = 'delphi_consensus'
        row_data['winner_model_uri'] = synth['organizer_model_uri']
        row_data['winner_provider'] = synth['organizer_provider']
        row_data['winner_role_name'] = synth['organizer_role_name']
        row_data['winner_role_description'] = synth['organizer_role_description']
        row_data['winner_writeup'] = synth['final_protocol']
        row_data[f'winner_{ANSWER_FIELD_NAME}'] = synth['final_answer']
        return row_data
    row_dicts = [row.to_dict() for _, row in input_df.iterrows()]
    with ThreadPoolExecutor(max_workers=CASE_CONCURRENCY) as pool:
        rows = list(tqdm(pool.map(_process_one_case, row_dicts), total=len(row_dicts), desc='Delphi pipeline'))
    return pd.DataFrame(rows)


In [ ]:
def build_participant_token_summary(result_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    organizer_pid = ORGANIZER_PARTICIPANT_ID
    for participant in PARTICIPANTS:
        pid = participant['id']

        def col(name: str) -> int:
            return int(result_df.get(name, pd.Series(dtype=int)).fillna(0).sum())
        r1_prompt = col(f'r1_prompt_tokens__{pid}')
        r1_completion = col(f'r1_completion_tokens__{pid}')
        r1_total = col(f'r1_total_tokens__{pid}')
        r2_prompt = col(f'r2_prompt_tokens__{pid}')
        r2_completion = col(f'r2_completion_tokens__{pid}')
        r2_total = col(f'r2_total_tokens__{pid}')
        r3_prompt = col(f'r3_prompt_tokens__{pid}')
        r3_completion = col(f'r3_completion_tokens__{pid}')
        r3_total = col(f'r3_total_tokens__{pid}')
        if pid == organizer_pid:
            org_prompt = col('organizer_after_r1_prompt_tokens') + col('organizer_after_r2_prompt_tokens') + col('final_synthesis_prompt_tokens')
            org_completion = col('organizer_after_r1_completion_tokens') + col('organizer_after_r2_completion_tokens') + col('final_synthesis_completion_tokens')
            org_total = col('organizer_after_r1_total_tokens') + col('organizer_after_r2_total_tokens') + col('final_synthesis_total_tokens')
        else:
            org_prompt = org_completion = org_total = 0
        rows.append({'participant_id': pid, 'provider': participant['provider'], 'model_uri': participant['model_uri'], 'role_name': participant['role_name'], 'r1_prompt_tokens': r1_prompt, 'r1_completion_tokens': r1_completion, 'r1_total_tokens': r1_total, 'r2_prompt_tokens': r2_prompt, 'r2_completion_tokens': r2_completion, 'r2_total_tokens': r2_total, 'r3_prompt_tokens': r3_prompt, 'r3_completion_tokens': r3_completion, 'r3_total_tokens': r3_total, 'organizer_prompt_tokens': org_prompt, 'organizer_completion_tokens': org_completion, 'organizer_total_tokens': org_total, 'all_prompt_tokens': r1_prompt + r2_prompt + r3_prompt + org_prompt, 'all_completion_tokens': r1_completion + r2_completion + r3_completion + org_completion, 'all_total_tokens': r1_total + r2_total + r3_total + org_total})
    return pd.DataFrame(rows).sort_values(['all_total_tokens', 'participant_id'], ascending=[False, True]).reset_index(drop=True)

def build_model_token_summary(participant_token_summary_df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [c for c in participant_token_summary_df.columns if c.endswith('_tokens')]
    return participant_token_summary_df.groupby(['provider', 'model_uri'], as_index=False)[numeric_cols].sum().sort_values(['all_total_tokens', 'model_uri'], ascending=[False, True]).reset_index(drop=True)


In [ ]:
_request_details = threading.local()

class PersistentSubsetCache(MutableMapping):

    def __init__(self, path, namespace):
        self.namespace = namespace
        self.lock = threading.RLock()
        self.connection = sqlite3.connect(path, timeout=60, check_same_thread=False)
        self.connection.execute('PRAGMA journal_mode=WAL')
        self.connection.executescript('\n            CREATE TABLE IF NOT EXISTS responses (\n                namespace TEXT NOT NULL, cache_key TEXT NOT NULL, payload TEXT NOT NULL,\n                PRIMARY KEY(namespace, cache_key));\n            CREATE TABLE IF NOT EXISTS usage_events (\n                event_id INTEGER PRIMARY KEY AUTOINCREMENT,\n                namespace TEXT NOT NULL, cache_key TEXT NOT NULL, model_uri TEXT NOT NULL,\n                prompt_tokens INTEGER NOT NULL, completion_tokens INTEGER NOT NULL,\n                total_tokens INTEGER NOT NULL, reasoning_tokens INTEGER,\n                finish_reason TEXT, max_tokens INTEGER, cost_rub REAL NOT NULL,\n                timestamp_utc TEXT NOT NULL);\n        ')
        self.connection.commit()

    def __getitem__(self, key):
        with self.lock:
            row = self.connection.execute('SELECT payload FROM responses WHERE namespace=? AND cache_key=?', (self.namespace, key)).fetchone()
        if row is None:
            raise KeyError(key)
        return json.loads(row[0])

    def __setitem__(self, key, value):
        details = getattr(_request_details, 'latest', None)
        with self.lock:
            self.connection.execute('INSERT OR REPLACE INTO responses VALUES (?, ?, ?)', (self.namespace, key, json.dumps(value, ensure_ascii=False)))
            if details is not None:
                self._insert_usage(key, details)
                _request_details.latest = None
            self.connection.commit()

    def __delitem__(self, key):
        with self.lock:
            cursor = self.connection.execute('DELETE FROM responses WHERE namespace=? AND cache_key=?', (self.namespace, key))
            self.connection.commit()
        if not cursor.rowcount:
            raise KeyError(key)

    def __iter__(self):
        with self.lock:
            keys = [row[0] for row in self.connection.execute('SELECT cache_key FROM responses WHERE namespace=?', (self.namespace,))]
        return iter(keys)

    def __len__(self):
        with self.lock:
            return self.connection.execute('SELECT COUNT(*) FROM responses WHERE namespace=?', (self.namespace,)).fetchone()[0]

    def clear(self):
        with self.lock:
            self.connection.execute('DELETE FROM responses WHERE namespace=?', (self.namespace,))
            self.connection.commit()

    def _insert_usage(self, key, details):
        model = details['model_uri'].split('/')[-2]
        prices = PRICES_RUB_PER_1000[model]
        cost = (details['prompt_tokens'] * prices['input'] + details['completion_tokens'] * prices['output']) / 1000
        self.connection.execute('INSERT INTO usage_events\n            (namespace,cache_key,model_uri,prompt_tokens,completion_tokens,total_tokens,\n             reasoning_tokens,finish_reason,max_tokens,cost_rub,timestamp_utc)\n            VALUES (?,?,?,?,?,?,?,?,?,?,?)', (self.namespace, key, details['model_uri'], details['prompt_tokens'], details['completion_tokens'], details['total_tokens'], details.get('reasoning_tokens'), details.get('finish_reason'), details.get('max_tokens'), cost, datetime.now(timezone.utc).isoformat()))

    def total_spend(self):
        with self.lock:
            return float(self.connection.execute('SELECT COALESCE(SUM(cost_rub),0) FROM usage_events').fetchone()[0])

    def usage_frame(self):
        with self.lock:
            return pd.read_sql_query('SELECT * FROM usage_events WHERE namespace=? ORDER BY event_id', self.connection, params=[self.namespace])

    def close(self):
        self.connection.close()
_original_create = openai_client.chat.completions.create

def create_with_usage_observation(**kwargs):
    if isinstance(llm_cache, PersistentSubsetCache):
        used = llm_cache.total_spend()
        if used >= BUDGET_LIMIT_RUB - BUDGET_SAFETY_BUFFER_RUB:
            raise RuntimeError(f'Local recorded spend is {used:.2f} RUB; budget safety threshold reached. Cached unfinished work is retained.')
    response = _original_create(**kwargs)
    usage = _extract_openai_usage(getattr(response, 'usage', None))
    usage_object = getattr(response, 'usage', None)
    completion_details = getattr(usage_object, 'completion_tokens_details', None)
    choices = getattr(response, 'choices', None) or []
    _request_details.latest = {'model_uri': kwargs['model'], **usage, 'reasoning_tokens': getattr(completion_details, 'reasoning_tokens', None), 'finish_reason': getattr(choices[0], 'finish_reason', None) if choices else None, 'max_tokens': kwargs.get('max_tokens')}
    return response
openai_client.chat.completions.create = create_with_usage_observation
_original_call_llm = call_llm

def call_llm_with_model_limits(provider, model_uri, system_text, user_text, temperature, max_tokens):
    if isinstance(llm_cache, PersistentSubsetCache) and llm_cache.total_spend() >= BUDGET_LIMIT_RUB - BUDGET_SAFETY_BUFFER_RUB:
        raise RuntimeError('Recorded-spend safety threshold reached. Resume after reviewing usage and the budget setting.')
    model = model_uri.split('/')[-2]
    effective_limit = MODEL_MAX_OUTPUT_TOKENS.get(model, max_tokens)
    _request_details.latest = None
    return _original_call_llm(provider, model_uri, system_text, user_text, temperature, effective_limit)
call_llm = call_llm_with_model_limits


In [ ]:
SOURCE_SHA256 = '06c54117679c01e76945565f6a9d9364a19a0f5915cbf355064af50c64870e5e'
SYSTEM_ID = 'delphi_gpt120_qwen36_deepseekv4'
PREDICTION_COLUMN = 'winner_selected_option'
ORIGINAL_PIPELINE = run_delphi_pipeline

def ORIGINAL_SUMMARIES(frame):
    participants = build_participant_token_summary(frame)
    return [('participant_token_summary_df', participants), ('model_token_summary_df', build_model_token_summary(participants))]
RUN_CONFIGURATION = {'system': SYSTEM_ID, 'source_sha256': SOURCE_SHA256, 'participants': PARTICIPANTS, 'organizer': ORGANIZER_PARTICIPANT_ID, 'max_rounds': MAX_DELPHI_ROUNDS, 'early_stop': EARLY_STOP_ON_UNANIMITY, 'temperatures': [TEMPERATURE_ROUND1, TEMPERATURE_ROUND2, TEMPERATURE_ROUND3, TEMPERATURE_ORGANIZER, TEMPERATURE_FINAL_SYNTHESIS], 'model_output_limits': MODEL_MAX_OUTPUT_TOKENS, 'reasoning_mode': 'provider_default'}


In [ ]:
def atomic_json(path, content):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(content, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def atomic_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, sep=';', index=False, encoding='utf-8-sig')
    temporary.replace(path)

def wilson_interval(correct, n):
    z = 1.959963984540054
    proportion = correct / n
    denominator = 1 + z * z / n
    center = (proportion + z * z / (2 * n)) / denominator
    radius = z * math.sqrt(proportion * (1 - proportion) / n + z * z / (4 * n * n)) / denominator
    return [center - radius, center + radius]

def run_subset_sequence():
    global llm_cache, df, result_df
    output_directory = RUNTIME_DIR / SYSTEM_ID
    output_directory.mkdir(parents=True, exist_ok=True)
    completed_metrics = []
    for subset_id in SUBSET_ORDER:
        specification = SUBSETS[subset_id]
        df = load_subset(subset_id)
        signature = hashlib.sha256(json.dumps({'configuration': RUN_CONFIGURATION, 'input_sha256': specification['sha256']}, sort_keys=True).encode()).hexdigest()
        namespace = f'{SYSTEM_ID}/{subset_id}/{signature}'
        llm_cache = PersistentSubsetCache(LEDGER_PATH, namespace)
        output_csv = output_directory / f'{subset_id}.csv'
        metrics_json = output_directory / f'{subset_id}.metrics.json'
        checkpoint_csv = output_directory / f'{subset_id}.partial.csv'
        checkpoint_json = output_directory / f'{subset_id}.partial.json'
        print(f"\n{subset_id}: n={len(df)}; split={specification['split']}")
        if output_csv.exists() and (not metrics_json.exists()):
            if not (checkpoint_csv.exists() and checkpoint_json.exists()):
                llm_cache.close()
                raise RuntimeError('Incomplete final output with no recovery checkpoint; retain the files for inspection.')
            print('Resuming interrupted finalization from the checkpoint.')
        elif output_csv.exists() or metrics_json.exists():
            if not output_csv.exists():
                llm_cache.close()
                raise RuntimeError('Final metrics exist but their result CSV is missing; retain the files for inspection.')
            saved = json.loads(metrics_json.read_text(encoding='utf-8'))
            if saved['configuration_signature'] != signature or saved['output_sha256'] != sha256_file(output_csv):
                llm_cache.close()
                raise ValueError('Existing result has a different configuration/checksum. Choose a new RUN_TAG for a new experiment.')
            if SKIP_COMPLETED:
                llm_cache.clear()
                llm_cache.close()
                completed_metrics.append(saved)
                print('Already complete:', output_csv)
                continue
            llm_cache.close()
            raise ValueError('Use a new RUN_TAG to repeat a completed experiment without overwriting it.')
        accumulated = pd.DataFrame()
        if checkpoint_csv.exists():
            if not checkpoint_json.exists():
                llm_cache.close()
                raise ValueError('Checkpoint metadata is missing.')
            checkpoint = json.loads(checkpoint_json.read_text(encoding='utf-8'))
            if checkpoint['configuration_signature'] != signature:
                llm_cache.close()
                raise ValueError('Checkpoint configuration differs; use a new RUN_TAG.')
            accumulated = pd.read_csv(checkpoint_csv, sep=';', encoding='utf-8-sig', dtype={'case_id': str})
            if accumulated['case_id'].duplicated().any() or not set(accumulated['case_id']).issubset(set(df['case_id'])):
                llm_cache.close()
                raise ValueError('Invalid checkpoint case IDs.')
        seen = set(accumulated['case_id']) if len(accumulated) else set()
        remaining = df.loc[~df['case_id'].isin(seen)].copy()
        atomic_json(checkpoint_json, {'configuration_signature': signature, 'completed_cases': len(accumulated)})
        try:
            for start in range(0, len(remaining), CHECKPOINT_BATCH_SIZE):
                batch = remaining.iloc[start:start + CHECKPOINT_BATCH_SIZE].copy()
                batch_result = ORIGINAL_PIPELINE(batch)
                accumulated = pd.concat([accumulated, batch_result], ignore_index=True, sort=False)
                atomic_csv(accumulated, checkpoint_csv)
                atomic_json(checkpoint_json, {'configuration_signature': signature, 'completed_cases': len(accumulated)})
                print(f'Checkpoint {len(accumulated)}/{len(df)}; recorded combined spend {llm_cache.total_spend():.2f} RUB')
            if set(accumulated['case_id']) != set(df['case_id']):
                raise ValueError('Final results do not cover the intended sample.')
            result_df = df[['case_id']].merge(accumulated, on='case_id', how='left', validate='one_to_one')
            result_df[PREDICTION_COLUMN] = result_df[PREDICTION_COLUMN].fillna('')
            result_part = result_df.copy()
            result_part['table_source'] = 'result_df'
            parts = [result_part]
            for source_name, frame in ORIGINAL_SUMMARIES(result_df):
                frame = frame.copy()
                frame['table_source'] = source_name
                parts.append(frame)
            combined_output = pd.concat(parts, ignore_index=True, sort=False)
            combined_output['subset_id'] = subset_id
            atomic_csv(combined_output, output_csv)
            usage = llm_cache.usage_frame()
            atomic_csv(usage, output_directory / f'{subset_id}.usage.csv')
            prediction = result_df[PREDICTION_COLUMN].fillna('').astype(str).str.strip().str.upper()
            valid = prediction.isin(ANSWER_LABELS)
            correct = int((valid & prediction.eq(result_df['correct_option_letter'])).sum())
            n = len(result_df)
            metrics = {'system_id': SYSTEM_ID, 'subset_id': subset_id, 'n': n, 'correct': correct, 'invalid_answers': int((~valid).sum()), 'accuracy': correct / n, 'accuracy_wilson_95ci': wilson_interval(correct, n), 'split': specification['split'], 'source_url': specification['source_url'], 'configuration_signature': signature, 'configuration': RUN_CONFIGURATION, 'input_sha256': specification['sha256'], 'output_sha256': sha256_file(output_csv), 'billed_prompt_tokens': int(usage['prompt_tokens'].sum()) if len(usage) else 0, 'billed_completion_tokens': int(usage['completion_tokens'].sum()) if len(usage) else 0, 'recorded_cost_rub': float(usage['cost_rub'].sum()) if len(usage) else 0.0, 'length_limited_calls': int(usage['finish_reason'].eq('length').sum()) if len(usage) else 0, 'price_source': PRICE_SOURCE, 'cache_policy': 'persist during the subset; clear responses after successful final write'}
            atomic_json(metrics_json, metrics)
            llm_cache.clear()
            completed_metrics.append(metrics)
            print(f'Saved {output_csv}; accuracy={correct}/{n} ({correct / n:.3%}); cache cleared.')
        finally:
            llm_cache.close()
    summary = pd.DataFrame([{key: record[key] for key in ['system_id', 'subset_id', 'n', 'correct', 'invalid_answers', 'accuracy', 'recorded_cost_rub', 'length_limited_calls']} for record in completed_metrics])
    atomic_csv(summary, output_directory / 'all_subsets_summary.csv')
    return summary
summary_df = run_subset_sequence()
summary_df
